In [57]:
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig
from vistiq.io import ImageLoader, ImageLoaderConfig
from vistiq.preprocess import ResizeConfig, Resize, DoG, DoGConfig, StackProcessorConfig, StackProcessor, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.utils import ArrayIteratorConfig 
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector
from skimage import filters, exposure, segmentation, draw
import supervision as sv

import stackview
import os
import numpy as np
from joblib import Parallel, delayed
import math
import logging

In [58]:
logger = logging.getLogger(__name__)

# Functions and Classes

In [59]:
from typing import Any, Union, Optional, Literal, Dict, Tuple, List

In [60]:
def box_iou_batch_3d(
    boxes_true: np.typing.NDArray[np.number],
    boxes_detection: np.typing.NDArray[np.number],
    overlap_metric: Literal["IOU", "IOS"] = "IOU"
) -> np.ndarray[np.float32]:
    """
    Adapted for 3d from https://github.com/roboflow/supervision/blob/develop/src/supervision/detection/utils/iou_and_nms.py
    
    Compute pairwise overlap scores between batches of bounding boxes.

    Supports standard IOU (intersection-over-union) and IOS
    (intersection-over-smaller-area) metrics for all `boxes_true` and
    `boxes_detection` pairs. Returns a matrix of overlap values in range
    `[0, 1]`, matching each box from the first batch to each from the second.

    Args:
        boxes_true: Array of reference boxes in
            shape `(N, 4)` as `(x_min, y_min, z_min, x_max, y_max, z_max)`.
        boxes_detection: Array of detected boxes in
            shape `(M, 4)` as `(x_min, y_min, z_min, x_max, y_max, z_min)`.
        overlap_metric: Overlap type.
            Use `OverlapMetric.IOU` for intersection-over-union,
            `OverlapMetric.IOS` for intersection-over-smaller-area.
            Defaults to `OverlapMetric.IOU`.

    Returns:
        Overlap matrix of shape `(N, M)`, where entry
            `[i, j]` is the overlap between `boxes_true[i]` and
            `boxes_detection[j]`.

    Raises:
        ValueError: If `overlap_metric` is not IOU or IOS.

    Examples:
        ```pycon
        >>> import numpy as np
        >>> import supervision as sv
        >>> boxes_true = np.array([
        ...     [100, 100, 200, 200],
        ...     [300, 300, 400, 400]
        ... ])
        >>> boxes_detection = np.array([
        ...     [150, 150, 250, 250],
        ...     [320, 320, 420, 420]
        ... ])
        >>> sv.box_iou_batch_3d(
        ...     boxes_true, boxes_detection, overlap_metric=sv.OverlapMetric.IOU
        ... )
        array([[0.14285..., 0.        ],
               [0.        , 0.47058...]], dtype=float32)
        >>> sv.box_iou_batch(
        ...     boxes_true, boxes_detection, overlap_metric=sv.OverlapMetric.IOS
        ... )
        array([[0.25, 0.  ],
               [0.  , 0.64]], dtype=float32)

        ```
    """
    #overlap_metric = OverlapMetric.from_value(overlap_metric)
    x_min_true, y_min_true, z_min_true, x_max_true, y_max_true, z_max_true = boxes_true.T
    x_min_det, y_min_det, z_min_det, x_max_det, y_max_det, z_max_det = boxes_detection.T
    count_true, count_det = boxes_true.shape[0], boxes_detection.shape[0]

    if count_true == 0 or count_det == 0:
        return cast(
            np.typing.NDArray[np.float32], np.empty((count_true, count_det), dtype=np.float32)
        )

    x_min_inter = np.empty((count_true, count_det), dtype=np.float32)
    x_max_inter = np.empty_like(x_min_inter)
    y_min_inter = np.empty_like(x_min_inter)
    y_max_inter = np.empty_like(x_min_inter)
    z_min_inter = np.empty_like(x_min_inter)
    z_max_inter = np.empty_like(x_min_inter)

    np.maximum(x_min_true[:, None], x_min_det[None, :], out=x_min_inter)
    np.minimum(x_max_true[:, None], x_max_det[None, :], out=x_max_inter)
    np.maximum(y_min_true[:, None], y_min_det[None, :], out=y_min_inter)
    np.minimum(y_max_true[:, None], y_max_det[None, :], out=y_max_inter)
    np.maximum(z_min_true[:, None], z_min_det[None, :], out=z_min_inter)
    np.minimum(z_max_true[:, None], z_max_det[None, :], out=z_max_inter)

    # we reuse x_max_inter and y_max_inter to store inter_w, inter_h and inter_d
    np.subtract(x_max_inter, x_min_inter, out=x_max_inter)  # inter_w
    np.subtract(y_max_inter, y_min_inter, out=y_max_inter)  # inter_h
    np.subtract(z_max_inter, z_min_inter, out=z_max_inter)  # inter_d
    np.clip(x_max_inter, 0.0, None, out=x_max_inter)
    np.clip(y_max_inter, 0.0, None, out=y_max_inter)
    np.clip(z_max_inter, 0.0, None, out=z_max_inter)

    area_inter = x_max_inter * y_max_inter * z_max_inter # inter_w * inter_h * inter_d

    area_true = (x_max_true - x_min_true) * (y_max_true - y_min_true) * (z_max_true - z_min_true)
    area_det = (x_max_det - x_min_det) * (y_max_det - y_min_det)  * (z_max_det - z_min_det)

    if overlap_metric == "IOU":
        area_norm = area_true[:, None] + area_det[None, :] - area_inter
    elif overlap_metric == "IOS":
        area_norm = np.minimum(area_true[:, None], area_det[None, :])
    else:
        raise ValueError(
            f"overlap_metric {overlap_metric} is not supported, "
            "only 'IOU' and 'IOS' are supported"
        )

    out: np.ndarray[np.float32] = np.zeros_like(area_inter, dtype=np.float32)
    np.divide(area_inter, area_norm, out=out, where=area_norm > 0)
    return out

In [61]:
def ac_3d(img, init=None, evolve_init=False, dtype="auto", out_max=1, points=100, margin=5, **kwargs):
    if img.ndim == 2:
        img = np.expand_dims(img, axis=0) 

    shape_2d = img.shape[-2:]
    if init is None:
        logger.info(f"creating initial rectangular snake with {points} points and a margin of {margin}")
        r, c = shape_2d
        init = np.array([
            np.concatenate([np.linspace(margin, c-2*margin, points), np.full(points, c-margin), 
                            np.linspace(c-2*margin, margin, points), np.full(points, margin)]),
            np.concatenate([np.full(points, margin), np.linspace(margin, r-2*margin, points), 
                            np.full(points, r-margin), np.linspace(r-2*margin, margin, points)])
        ]).T
    logger.debug(f"img.dtype={img.dtype}, img.shape={img.shape}, init.shape={init.shape}")
    if dtype == "auto":
        dtype = img.dtype
    if evolve_init:
        isnake_mask = np.zeros(img.shape, dtype=dtype)
        isnake_points = np.zeros((img.shape[0], init.shape[0], init.shape[1],), dtype="float64")
        for z in reversed(range(img.shape[0])):
            isnake = segmentation.active_contour(img[z], init, **kwargs)
            isnake_mask[z] = draw.polygon2mask(shape_2d, isnake).astype(dtype)*out_max
            init = isnake
            isnake_points[z] = isnake
    else:
        isnake_points = np.array(Parallel(n_jobs=-1, verbose=10)(delayed(segmentation.active_contour)(i, init, **kwargs) for i in img))
        isnake_mask = np.array(Parallel(n_jobs=-1, verbose=10)(delayed(draw.polygon2mask)(shape_2d, isnake) for isnake in isnake_points))
        isnake_mask = isnake_mask.astype(dtype)*out_max
    return isnake_mask.squeeze(), isnake_points.squeeze()


In [62]:
def labels_to_masks(labels):
    label_values = (v for v in np.unique(labels) if v > 0)
    masks = []
    for value in label_values:
        mask = labels == value
        masks.append(mask)
    return np.array(masks)

In [63]:
def group_bboxes(bboxes, divisor=1, threshold=0.5):
    
    def in_groups(item, groups):
        for g in groups:
            if item in g:
                return True
        return False
    
    xyxy = np.mod(bboxes, divisor)
    #print (xyxy[:7])
    if len(bboxes[0]) == 4:
        iou_matrix = sv.box_iou_batch(xyxy, xyxy, overlap_metric=sv.OverlapMetric.IOU)
    elif len(bboxes[0]) == 6:
        iou_matrix = box_iou_batch_3d(xyxy, xyxy, overlap_metric="IOU")
    #print (iou_matrix)
    iou_matrix = np.triu(iou_matrix, k=1)
    pairs = np.argwhere(iou_matrix > threshold)
    
    groups = []
    for i, pair in enumerate(pairs):
        p0 = pair[0]
        #print (i, p0, p1, iou_matrix[p0, p1])
        if not in_groups(p0, groups):
            pairs_with_p0 = np.unique(np.array([p for p in pairs if p[0] == p0]).flatten())
            logger.info(f"Creating new group with {pairs_with_p0}")
            groups.append(pairs_with_p0)
    return groups

In [64]:
def label_grouped_mask(mask, groups:list[np.ndarray], threshold=1):
    labels = []
    th = math.prod(tile_factor)//2
    for label_value, g in enumerate(groups, 1):
        label_array = (mask[g].sum(axis=0)>threshold)*label_value
        labels.append(label_array)
    labels = np.sum(np.array(labels), axis=0).astype("uint16")
    return labels

In [65]:
class UntilerConfig(StackProcessorConfig):

    factor: Tuple[int,...] = (1,1)

In [66]:
class Untiler(StackProcessor):

    def __init__(self, config: UntilerConfig):
        super().__init__(config)

    def _process_slice(self, slice: np.ndarray, metadata: Optional[dict[str, Any]] = None, **kwargs) -> np.ndarray:
        factor = self.config.factor
        assert len(factor) == 2
        import math
        resized = tuple(size_in // f for size_in, f in zip(slice.shape[-2:], factor))
    
        vs = np.array(np.split(slice, factor[-1], axis=-1))
        hs = np.array(np.split(vs, factor[-2], axis=-2))
        untiled = hs.reshape((math.prod(factor), *slice.shape[:-2], *resized))
        logger.info(f"factor={factor}, array.shape={slice.shape}, vs.shape={vs.shape}, hs.shape={hs.shape}, untiled.shape={untiled.shape}")
        return untiled        

In [67]:
class FuncProcessorConfig(StackProcessorConfig):

    iterator_config: ArrayIteratorConfig = ArrayIteratorConfig(slice_def=())
    strict_axis: bool = True
    dtype: Literal[np.uint8, np.uint16, np.uint32, np.uint64, np.float32, np.float64, bool] = None
    func: Any = None
    args: List[Any] = []
    kwargs: Dict[str, Any] = {}

In [68]:
class FuncProcessor(StackProcessor):

    def __init__(self, config:FuncProcessorConfig):
        super().__init__(config)

    def _process_slice(self, slice: np.ndarray, metadata: Optional[dict[str, Any]] = None, **kwargs) -> np.ndarray:
        func = self.config.func
        args = self.config.args
        kwargs = self.config.kwargs.copy()
        if "axis" in kwargs:
            axis_letters = kwargs["axis"]
            axis_indices = tuple([self._axis_index(metadata,letter) if isinstance(letter, str) else letter for letter in axis_letters])
            axis_indices=tuple([i for i in axis_indices if i is not None])
            kwargs["axis"] = axis_indices
            logger.info(f"Mapped axis letters {axis_letters} to axis indices {axis_indices}")
        results = func(slice, *args, **kwargs)
        if self.config.dtype is not None and isinstance(results, np.ndarray):
            results = results.astype(self.config.dtype)
            logger.info(f"Converted results to {results.dtype}")
        return results

In [69]:
class RescaleConfig(PreprocessorConfig):

    low: float = 0.0
    high: float = 100.0
    

In [70]:
class Rescale(Preprocessor):

    def __init__(self, config:RescaleConfig):
        super().__init__(config)

    def _process_slice(self, slice: np.ndarray, metadata: Optional[dict[str, Any]] = None, **kwargs) -> np.ndarray:
        plow, phigh = np.percentile(slice, (self.config.low, self.config.high))
        logger.debug(plow, phigh)
        scaled = exposure.rescale_intensity(slice, out_range=self.config.dtype, in_range=(plow,phigh))
        return scaled        

In [71]:
class TilerConfig(StackProcessorConfig):

    factor: Tuple[int,...]
    pad_width: Union[int, Tuple[Tuple[int,int]], Dict[int,Union[int,Tuple[int,int]]]] = None
    pad_kwargs: Dict[str, Any] = {"mode": "constant", "constant_values": 0}
    alt_flip: bool = False
    iterator_config: ArrayIteratorConfig = ArrayIteratorConfig(slice_def=())

In [72]:
class Tiler(StackProcessor):

    def __init__(self, config:TilerConfig):
        super().__init__(config)

    def _process_slice(self, slice: np.ndarray, metadata: Optional[dict[str, Any]] = None, **kwargs) -> np.ndarray:
        factor = self.config.factor
        if self.config.alt_flip:
            import math
            orig_y, orig_x = slice.shape[-2:]
            top_row = np.hstack((slice, np.fliplr(slice)))
            bottom_row = np.flipud(top_row)
            block = np.vstack((top_row, bottom_row))
            n_x = int(math.ceil(factor[-1]/2))
            n_y = int(math.ceil(factor[-2]/2))
            tiled = np.tile(block, (n_y, n_x))
            start_x = tiled.shape[-1] - (factor[-1] * orig_x)
            start_y = tiled.shape[-1] - (factor[-2] * orig_x)
            tiled = tiled[..., start_y:, start_x:]
            print (slice.shape, orig_y, orig_x, n_x, n_y, tiled.shape)
        else:
            tiled = np.tile(slice, factor) 
        return tiled

    def run(self, stack, *args, metadata: Optional[dict[str, Any]] = None, **kwargs):
        if self.config.pad_width is not None:
            stack = np.pad(stack, pad_width=self.config.pad_width, **self.config.pad_kwargs)
        return super().run(stack, *args, metadata=metadata, **kwargs)


# Load Image

In [73]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"

scene_index = 0

embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [74]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=True, 
    #substack="C:1"
)
img, metadata = ImageLoader(ilc).run(path)
metadata

2026-05-22 13:17:33,574 - INFO - Loading image from: /standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif
2026-05-22 13:17:33,829 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-05-22 13:17:33,928 - INFO - Loaded image: /standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif scene=0 -> shape=(3, 93, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-05-22 13:17:33,929 - INFO - Loaded image with shape: (3, 93, 512, 512), dtype: uint8
2026-05-22 13:17:33,930 - INFO - Finished in state Completed()


{'scene_index': 0,
 'dim_order': 'CZYX',
 'axes': ['C', 'Z', 'Y', 'X'],
 'channel_names': ['Scrib', 'EdU', 'Dpn'],
 'channel_axis': 0,
 'shape': (3, 93, 512, 512),
 'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
 'pixel_unit': 'um',
 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
 'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)}

# Preprocess

In [75]:
scfg = RescaleConfig(
    low=2, 
    high=98, 
    dtype=np.uint8, 
    iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
)
simg,_ = Rescale(scfg).run(img, metadata=metadata, verbose=1)

2026-05-22 13:17:39,939 - INFO - Running preprocessor Rescale, on stack of type uint8, True
2026-05-22 13:17:40,063 - INFO - Running Rescale with config: classname='RescaleConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=<class 'numpy.uint8'> low=2.0 high=98.0
2026-05-22 13:17:40,063 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 13:17:40,063 - INFO - Using Parallel with n_jobs=-1 for 3 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.6s finished
2026-05-22 13:17:40,628 - INFO - Reshaped results to shape (3, 93, 512

In [76]:
gcfg = FuncProcessorConfig(
    func=filters.gaussian,
    kwargs={"sigma": 1.0},
    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1))
)
nimg, _ = FuncProcessor(gcfg).run(simg, verbose=1)    

2026-05-22 13:17:41,379 - INFO - Running FuncProcessor with config: classname='FuncProcessorConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=True dtype=None func=<function gaussian at 0x7f2d026c6980> args=[] kwargs={'sigma': 1.0}
2026-05-22 13:17:41,380 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 13:17:41,381 - INFO - Using Parallel with n_jobs=-1 for 279 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done 279 out of 279 | elapsed:    1.2s finished
2026-05-22 13:17:42,632 - INFO - Reshaped results to shape (3, 93, 512, 512)
2026-05-22 13:17:42,655 - INFO - Fin

In [77]:
gamma = 0.2
enimg = np.array([exposure.rescale_intensity(exposure.adjust_sigmoid(exposure.adjust_gamma(i, gamma=gamma)),out_range="uint8") for i in nimg])
print (np.max(enimg))
#emimg = exposure.rescale_intensity(filters.gaussian(exposure.adjust_sigmoid(exposure.adjust_gamma(mimg, gamma=gamma)), sigma=5.0), out_range="uint8")

255


In [78]:
stackview.slice(np.concatenate([simg, exposure.rescale_intensity(nimg, out_range="uint8"), enimg], axis=-1))

In [79]:
# Project all channels to one.

pcfg = FuncProcessorConfig(
    func=np.max, 
    kwargs={"axis":("C")}, 
    strict_axis=False,
    dtype=np.uint16,
)
c_img, c_metadata = FuncProcessor(pcfg).run(enimg, metadata=metadata)
metadata, c_metadata

2026-05-22 13:17:53,836 - INFO - Running FuncProcessor with config: classname='FuncProcessorConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=False dtype=<class 'numpy.uint16'> func=<function max at 0x7f2d2c176eb0> args=[] kwargs={'axis': 'C'}
2026-05-22 13:17:53,837 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 13:17:53,838 - INFO - Mapped axis letters C to axis indices (0,)
2026-05-22 13:17:53,850 - INFO - Converted results to uint16
2026-05-22 13:17:53,850 - INFO - Dropping axes. New axes: ['Z', 'Y', 'X']
2026-05-22 13:17:53,851 - INFO - Updating metadata with new shape ratio: [1. 1. 1.]
2026-05-22 13:17:53,852 - INFO - Metadata upd

({'scene_index': 0,
  'dim_order': 'CZYX',
  'axes': ['C', 'Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (3, 93, 512, 512),
  'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)})

In [80]:
stackview.slice(c_img, continuous_update=True)

# Rough Mask of Projection to inform Lobe Segmentation

In [81]:
#c_mask, c_points = ac_3d(proj, points=100, margin=0, out_max=255, alpha=0.015, beta=0.1, gamma=0.001, w_edge=1.75, boundary_condition="periodic") # beta=0.1, w_edge=1.75
#logging.info(f"{proj.dtype}, {c_mask.dtype}, {np.max(proj)}, {np.max(c_mask)}")

In [82]:
#stackview.slice(np.concatenate([proj, animg[40], c_mask, (proj-0.2*c_mask)], axis=-1))

In [83]:
#isnake_img,_ = ac_3d(animg[::4], init=c_points, out_max=255, alpha=0.015, beta=0.1, gamma=0.001, w_edge=1.75, boundary_condition="periodic") #w_edge=1.75

In [84]:
#stackview.slice(np.concatenate([img[0,::4]+0.2*isnake_img, animg[::4]+0.2*isnake_img, isnake_img], axis=-1))

# Detect Tissue boundaries with MicroSAM (resampling)

## Resize

In [85]:
factor = 3
width = c_img.shape[-1]//factor
padding = 10

rcfg = ResizeConfig(width=width)
r_img, r_metadata = Resize(rcfg).run(c_img, metadata=c_metadata, verbose=1)
c_metadata, r_metadata

2026-05-22 13:18:02,406 - INFO - RESIZING stack from (93, 512, 512) to [93, 170, 170]
2026-05-22 13:18:02,489 - INFO - Running preprocessor Resize, on stack of type uint16, True
2026-05-22 13:18:02,571 - INFO - Running Resize with config: classname='ResizeConfig' package='vistiq.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=(93, 170, 170) output_axes=None recompute_scale=True squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=None width=170 height=None order=1 preserve_range=True anti_aliasing=True
2026-05-22 13:18:02,571 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 13:18:02,572 - INFO - Using Parallel with n_jobs=-1 for 93 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Paralle

({'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 170, 170),
  'dims': <Dimensions [Z: 93, Y: 170, X: 170]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)})

In [86]:
stackview.orthogonal(r_img)

## Z-Project

In [35]:
fcfg = FuncProcessorConfig(
    func=np.mean, 
    kwargs={"axis":("Z")},
)
proj, p_metadata = FuncProcessor(fcfg).run(r_img, metadata=r_metadata)

2026-05-22 12:47:16,049 - INFO - Running FuncProcessor with config: classname='FuncProcessorConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=True dtype=None func=<function mean at 0x7f2d2c1800f0> args=[] kwargs={'axis': 'Z'}
2026-05-22 12:47:16,049 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 12:47:16,050 - INFO - Mapped axis letters Z to axis indices (0,)
2026-05-22 12:47:16,052 - INFO - Dropping axes. New axes: ['Y', 'X']
2026-05-22 12:47:16,053 - INFO - Updating metadata with new shape ratio: [1. 1.]
2026-05-22 12:47:16,053 - INFO - Metadata updated in FuncProcessor: 4 key(s) changed
2026-05-22 12:47:16,054 - INFO -   axes: ['Z', 

In [36]:
tile_factor = (factor, factor)
tcfg = TilerConfig(factor=tile_factor, alt_flip=False, pad_width={-2:(0,padding),-1:(0,padding)})
t_proj,t_metadata = Tiler(tcfg).run(proj, metadata=p_metadata)
p_metadata, t_metadata

2026-05-22 12:47:16,663 - INFO - Running Tiler with config: classname='TilerConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3) pad_width={-2: (0, 10), -1: (0, 10)} pad_kwargs={'mode': 'constant', 'constant_values': 0} alt_flip=False
2026-05-22 12:47:16,664 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 12:47:16,665 - INFO - Updating metadata with new shape ratio: [0.33333333 0.33333333]
2026-05-22 12:47:16,665 - INFO - Metadata updated in Tiler: 2 key(s) changed
2026-05-22 12:47:16,666 - INFO -   shape: (93, 170) -> (309, 540)
2026-05-22 12:47:16,667 - INFO -   dims: <Dimensions [Y: 93, X: 170]> -> <Dimensions [Y: 309, X: 540]>
2026-05

({'scene_index': 0,
  'dim_order': 'YX',
  'axes': ['Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 170),
  'dims': <Dimensions [Y: 93, X: 170]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)},
 {'scene_index': 0,
  'dim_order': 'YX',
  'axes': ['Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (309, 540),
  'dims': <Dimensions [Y: 309, X: 540]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)})

In [37]:
rcfg = RegionFilterConfig(filters=[
    RangeFilter(
        RangeFilterConfig(
            attribute="circularity", range=(0.5,1.0)
        ),
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="aspect_ratio", range=(0.5,1.0)
        ),
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="cross_sectional_area", range=(1500,100000)
        ),
    )]

)
rf = RegionFilter(rcfg)

racfg = RegionAnalyzerConfig(
    properties=["area", "cross_sectional_area", "bbox", "aspect_ratio", "circularity"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe"
)
ra = RegionAnalyzer(racfg)

mcfg = MicroSAMSegmenterConfig(
    region_filter=rf, 
    region_analyzer=ra,
    embedding_path=embedding_path
)
p_mask,p_labels, p_results = MicroSAMSegmenter(mcfg).run(t_proj, metadata=t_metadata)

2026-05-22 12:47:17,199 - INFO - Labeller not provided, using default Labeller with connectivity=1 and region_filter=None
2026-05-22 12:47:17,199 - INFO - Segmenter config: classname=None package=None version=None command_group=None thresholder=None binary_processor=None labeller=Labeller(classname='LabellerConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='list' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None connectivity=1 region_filter=None) region_analyzer=RegionAnalyzer(classname='RegionAnalyzerConfig' package='vistiq.segment.analysis' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=No

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'numpy.ndarray'>


2026-05-22 12:47:25,784 - INFO - Finished in state Completed()


In [38]:
p_results.describe()

,area,bbox-0,bbox-1,bbox-2,bbox-3,circularity,aspect_ratio,cross_sectional_area
count,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000
mean,3705.610526,104.777778,218.444444,192.222222,289.222222,0.778201,0.662258,4099.985921
std,800.469873,88.053645,148.744841,89.639246,156.174404,0.081402,0.075806,885.661131
min,3020.798235,2.000000,41.000000,87.000000,108.000000,0.591179,0.598056,3342.291411
25%,3336.928283,5.000000,47.000000,89.000000,109.000000,0.765846,0.634872,3692.066093
50%,3489.683629,104.000000,229.000000,193.000000,290.000000,0.812754,0.651391,3861.078666
75%,3570.554106,206.000000,375.000000,295.000000,469.000000,0.825429,0.653408,3950.555910
max,5651.130933,207.000000,404.000000,296.000000,470.000000,0.847383,0.857349,6252.561379


In [39]:
stackview.blend(t_proj, p_labels, blend_factor=25, continuous_update=True)

# Consensus voting
Find matching regions based on IoU of bounding boxes, then create consensus masks (only consider pixels that show up in at least half of the masks).

In [40]:
logger.info(f"width={width}, p_labels.shape={p_labels.shape}, {p_labels.shape[-1]//factor}")
groups = group_bboxes(p_results[["bbox-1", "bbox-0", "bbox-3","bbox-2"]].to_numpy()-np.array((0,0,1,1)), divisor=p_labels.shape[-1]//factor, threshold=0.5)

ucfg = UntilerConfig(
    factor = tile_factor,
    iterator_config = ArrayIteratorConfig(slice_def=())
)

masks = labels_to_masks(p_labels)
untiled,_ = Untiler(ucfg).run(masks)
#untiled = untile(masks, tile_factor)
u_proj =  np.sum(untiled>0, axis=0)>0

labels = label_grouped_mask(u_proj, groups, threshold=math.prod(tile_factor)//2)
labels = labels[..., 0:labels.shape[-2]-padding, 0:labels.shape[-1]-padding]
stackview.blend(proj, labels, blend_factor=25, continuous_update=True)

2026-05-22 12:47:29,677 - INFO - width=170, p_labels.shape=(309, 540), 180
2026-05-22 12:47:29,678 - INFO - Creating new group with [0 1 2 8]
2026-05-22 12:47:29,679 - INFO - Creating new group with [6 7 8]
2026-05-22 12:47:29,696 - INFO - Running Untiler with config: classname='UntilerConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3)
2026-05-22 12:47:29,697 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 12:47:29,697 - INFO - factor=(3, 3), array.shape=(9, 309, 540), vs.shape=(3, 9, 309, 180), hs.shape=(3, 3, 9, 103, 180), untiled.shape=(9, 9, 103, 180)
2026-05-22 12:47:29,699 - INFO - Finished in state Completed()


# Segment tiled Z-stack

In [41]:
tr_img,tr_metadata = Tiler(tcfg).run(r_img, metadata=r_metadata)
r_metadata, tr_metadata

2026-05-22 12:47:31,718 - INFO - Running Tiler with config: classname='TilerConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3) pad_width={-2: (0, 10), -1: (0, 10)} pad_kwargs={'mode': 'constant', 'constant_values': 0} alt_flip=False
2026-05-22 12:47:31,719 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 12:47:31,753 - INFO - Updating metadata with new shape ratio: [1.         0.33333333 0.33333333]
2026-05-22 12:47:31,754 - INFO - Metadata updated in Tiler: 2 key(s) changed
2026-05-22 12:47:31,754 - INFO -   shape: (93, 170, 170) -> (170, 309, 540)
2026-05-22 12:47:31,755 - INFO -   dims: <Dimensions [Z: 93, Y: 170, X: 170]> -> <Dimensi

({'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 170, 170),
  'dims': <Dimensions [Z: 93, Y: 170, X: 170]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (170, 309, 540),
  'dims': <Dimensions [Z: 170, Y: 309, X: 540]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)})

In [42]:
mcfg = MicroSAMSegmenterConfig(
    region_filter=rf, 
    region_analyzer=RegionAnalyzer(
       RegionAnalyzerConfig(
            output_type="dataframe", 
            properties=["cross_sectional_area", "bbox", "volume", "aspect_ratio"],
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
        )
    ),
    do_regions=True,
    embedding_path=embedding_path,
    #iterator_config=ArrayIteratorConfig(slice_def=()),
)
tmask, tlabels, tresults = MicroSAMSegmenter(mcfg).run(tr_img, metadata=tr_metadata)

2026-05-22 12:47:32,529 - INFO - Labeller not provided, using default Labeller with connectivity=1 and region_filter=None
2026-05-22 12:47:32,530 - INFO - Segmenter config: classname=None package=None version=None command_group=None thresholder=None binary_processor=None labeller=Labeller(classname='LabellerConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='list' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None connectivity=1 region_filter=None) region_analyzer=RegionAnalyzer(classname='RegionAnalyzerConfig' package='vistiq.segment.analysis' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' til

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'numpy.ndarray'>


2026-05-22 12:52:22,981 - INFO - Finished in state Completed()
2026-05-22 12:52:23,400 - INFO - Finished in state Completed()
2026-05-22 12:52:23,556 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a1088a-6d77-7dbc-8000-d4902a3cc1f8/set_state "HTTP/1.1 201 Created"
2026-05-22 12:52:24,504 - INFO - Finished in state Completed()


In [48]:
stackview.blend(tr_img, tlabels, blend_factor=25, continuous_update=True)

In [49]:
tresults.describe()

,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,aspect_ratio,cross_sectional_area,volume
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,74.740741,110.629630,180.962963,133.888889,189.666667,290.481481,0.675281,5563.656893,236442.796941
std,56.100591,85.347772,155.394654,44.923296,85.468393,161.082246,0.057585,2597.282653,142386.657440
min,0.000000,0.000000,0.000000,64.000000,50.000000,56.000000,0.549246,2026.343251,29091.686048
25%,1.000000,0.000000,7.000000,74.000000,93.000000,144.000000,0.637228,2149.261486,46397.138834
50%,92.000000,103.000000,194.000000,164.000000,196.000000,299.000000,0.672546,6299.559527,299885.490103
75%,133.500000,206.000000,359.500000,168.000000,263.500000,434.000000,0.702090,7920.995652,364143.107573
max,136.000000,206.000000,397.000000,168.000000,299.000000,516.000000,0.851212,9033.586438,378833.935275


In [50]:
print (f"width={width}, tlabels.shape={tlabels.shape}, {tlabels.shape[-1]//factor}")
tgroups = group_bboxes(tresults[["bbox-2", "bbox-1", "bbox-0", "bbox-5", "bbox-4", "bbox-3"]].to_numpy()-np.array((0,0,0,1,1,1)), divisor=tlabels.shape[-1]//factor, threshold=0.5)

tmasks = labels_to_masks(tlabels)

ucfg = UntilerConfig(
    factor = tile_factor,
    iterator_config = ArrayIteratorConfig(slice_def=())
)
untiled,_ = Untiler(ucfg).run(tmasks)
tproj =  np.sum(untiled>0, axis=0)>0

labels = label_grouped_mask(tproj, tgroups, threshold=math.prod(tile_factor)//2)
print (labels.dtype)
cropped_height = labels.shape[-2]-padding
cropped_width = labels.shape[-1]-padding
cropped_labels = labels[..., 0:cropped_height, 0:cropped_width]

2026-05-22 12:56:06,340 - INFO - Creating new group with [1 2 3 8]
2026-05-22 12:56:06,341 - INFO - Creating new group with [4 5 8]
2026-05-22 12:56:06,341 - INFO - Creating new group with [10 21 26]
2026-05-22 12:56:06,341 - INFO - Creating new group with [11 16 18]
2026-05-22 12:56:06,342 - INFO - Creating new group with [12 16]
2026-05-22 12:56:06,343 - INFO - Creating new group with [17 19 23]
2026-05-22 12:56:06,344 - INFO - Creating new group with [24 25]


width=170, tlabels.shape=(170, 309, 540), 180


2026-05-22 12:56:08,211 - INFO - Running Untiler with config: classname='UntilerConfig' package='__main__' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3)
2026-05-22 12:56:08,212 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 12:56:08,509 - INFO - factor=(3, 3), array.shape=(27, 170, 309, 540), vs.shape=(3, 27, 170, 309, 180), hs.shape=(3, 3, 27, 170, 103, 180), untiled.shape=(9, 27, 170, 103, 180)
2026-05-22 12:56:08,512 - INFO - Finished in state Completed()


uint16


In [51]:
stackview.slice(tmasks)

In [56]:
ecfg = ResizeConfig(width=img.shape[-1], normalize=False, dtype=np.uint16)
labels, l_metadata = Resize(ecfg).run(cropped_labels, metadata=r_metadata)

stackview.blend(
    np.swapaxes(c_img.astype(np.uint16), 0, 1),
    labels.astype(np.uint64), 
    blend_factor=25 # Sets the transparency of the overlay (0.0 to 1.0)
)

2026-05-22 13:00:21,794 - INFO - RESIZING stack from (170, 93, 170) to [170, 280, 512]
2026-05-22 13:00:21,806 - INFO - Running preprocessor Resize, on stack of type uint16, True
2026-05-22 13:00:21,820 - INFO - Running Resize with config: classname='ResizeConfig' package='vistiq.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=(170, 280, 512) output_axes=None recompute_scale=True squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=<class 'numpy.uint16'> width=512 height=None order=1 preserve_range=True anti_aliasing=True
2026-05-22 13:00:21,822 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-22 13:00:21,822 - INFO - Using Parallel with n_jobs=-1 for 170 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurr

(512, 93, 512) (170, 280, 512)


# Save label

In [70]:
from vistiq.io import ImageWriterConfig, ImageWriter
imc = ImageWriterConfig()
outpath = ".".join(path.split(".")[:-1])+f"-thumbnail-{width}x{width}.tif"
ImageWriter(imc).run(rimg, outpath, metadata=rmetadata)

NameError: name 'rimg' is not defined